# 116 — Herramientas tipadas y efectos laterales

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

Una **herramienta tipada** es un contrato de tres partes: descripción semántica (cuándo
usarla y cuándo NO), **JSON Schema** de entrada (tipos, required, enums, rangos — se
valida ANTES de ejecutar) y contrato de salida/errores (el error estructurado es parte
de la interfaz: el agente lo observa y decide con él).

**Taxonomía de efectos:** pura/solo lectura → escritura reversible → escritura
irreversible → efecto externo distribuido. La clase de efecto determina qué controles
exige (reintento libre, registro, aprobación humana, auditoría).

### 🔁 Idempotencia y dry-run

**Idempotente:** `f(f(x)) = f(x)` — repetir la operación no multiplica el efecto
(`set_price(10)` sí; `add_units(+5)` no). Importa porque los agentes REINTENTAN y un
timeout no dice si el efecto se aplicó. Técnica estándar: **clave de idempotencia** —
el servidor registra las claves aplicadas y convierte duplicados en no-ops.

**Dry-run:** con `dry_run: true` la herramienta valida precondiciones y devuelve QUÉ
haría (diff/plan/costo) sin aplicar nada. Convierte un efecto irreversible en dos
pasos: uno observable y uno autorizado.

El laboratorio `agent` usa dos herramientas puras (`status`, `sum`): reintentables sin
riesgo — el caso base contra el que se mide todo lo demás.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Ejercicios

**Ejercicio 1 — Clasifica efectos.** Ejecuta `run_lab("agent", seed=116)` y confirma que
las dos herramientas de la traza son puras. Después clasifica estas seis en la taxonomía
(pura / reversible / irreversible / externa distribuida) y di si son idempotentes:
`get_balance()`, `set_status(id, "closed")`, `append_log(line)`, `send_email(to, body)`,
`delete_file(path)` (sin papelera), `upsert_config(key, value)`.

**Ejercicio 2 — Diseña el schema.** Escribe el JSON Schema de entrada para una
herramienta `refund_order(order_id, amount, reason, dry_run)` de una tienda: `order_id`
con patrón, `amount` con mínimo/máximo, `reason` como enum de 3 valores, `dry_run` con
default true. ¿Qué campo añadirías para que el reintento tras timeout sea seguro?

**Ejercicio 3 — Predice la salida.** Con stock `{"central": 3}`, predice la observación
de cada llamada a `transfer_inventory` (la del README): (a) `units=10, dry_run=true`;
(b) `units=3, dry_run=true`; (c) `units=3, dry_run=false, key="k1"`; (d) exactamente la
misma llamada (c) repetida. Escribe las cuatro observaciones esperadas.

**Ejercicio 4 — Simula la clave de idempotencia.** Implementa en Python un registro de
idempotencia: una función `apply(key, operation)` que ejecuta `operation()` solo si
`key` no fue aplicada y devuelve el resultado original en los duplicados. Demuestra con
un contador que dos `apply("k1", ...)` no duplican el efecto.

In [ ]:
# TODO: ejecuta run_lab("agent", seed=116)
# TODO: comprueba que el resultado incluya las claves 'kind' y 'evidence'
result = None


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 1: verifica el lab y clasifica
result = run_lab("agent", seed=116)
for paso in result["result"]["trace"]:
    print(paso["action"]["tool"], "->", paso["observation"])

clasificacion = {
    "get_balance":   {"efecto": "?", "idempotente": None},
    "set_status":    {"efecto": "?", "idempotente": None},
    "append_log":    {"efecto": "?", "idempotente": None},
    "send_email":    {"efecto": "?", "idempotente": None},
    "delete_file":   {"efecto": "?", "idempotente": None},
    "upsert_config": {"efecto": "?", "idempotente": None},
}


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicios 2 y 4
schema_refund = {
    "type": "object",
    "properties": {
        # completa: order_id (pattern), amount (minimum/maximum),
        # reason (enum de 3), dry_run (default true), ¿y el reintento seguro?
    },
    "required": [],
}

registro = {}
def apply(key, operation):
    # ejecuta operation() solo si key es nueva; duplicados -> resultado original
    pass


## Reflexión

1. Las dos herramientas del laboratorio son puras. ¿Qué tres mecanismos nuevos (schema,
   dry-run, idempotency_key) se vuelven obligatorios en cuanto una herramienta escribe
   estado, y qué riesgo concreto cubre cada uno?
2. ¿Por qué un timeout es el caso que separa "reintentar" de "reintentar con clave de
   idempotencia"? ¿Qué información NO te da un timeout?
3. Propón el error estructurado que debería devolver una herramienta `book_meeting`
   cuando la sala está ocupada, de modo que el siguiente thought pueda replantear sin
   intervención humana.